# Phase III: modification of PPI network with gene expression correlations

## Part 1: Create Mapping Dictionary from Ensembl IDs of proteins to HGNC symbols

In [1]:
import networkx as nx
import pandas as pd

In [ ]:
# Upload information table from STRING to map protein IDs to gene symbols
info_df = pd.read_csv("../data/STRING/9606.protein.info.v12.0.txt", sep="\t", usecols=[0,1])
print(info_df.shape)
info_df.head()

(19699, 2)


,#string_protein_id,preferred_name
0,9606.ENSP00000000233,ARF5
1,9606.ENSP00000000412,M6PR
2,9606.ENSP00000001008,FKBP4
3,9606.ENSP00000001146,CYP26B1
4,9606.ENSP00000002125,NDUFAF7


In [ ]:
# Check for empty rows
info_df.isna().value_counts()

#string_protein_id  preferred_name
False               False             19699
Name: count, dtype: int64

In [ ]:
# Check for duplicate HGNC symbols
info_df["preferred_name"].duplicated().value_counts()

preferred_name
False    19699
Name: count, dtype: int64

In [5]:
# Create mapping dict 
protein_id_to_symbol = dict(zip(info_df["#string_protein_id"],info_df["preferred_name"]))
print(protein_id_to_symbol)

{'9606.ENSP00000000233': 'ARF5', '9606.ENSP00000000412': 'M6PR', '9606.ENSP00000001008': 'FKBP4', '9606.ENSP00000001146': 'CYP26B1', '9606.ENSP00000002125': 'NDUFAF7', '9606.ENSP00000002165': 'FUCA2', '9606.ENSP00000002596': 'HS3ST1', '9606.ENSP00000002829': 'SEMA3F', '9606.ENSP00000003084': 'CFTR', '9606.ENSP00000003100': 'CYP51A1', '9606.ENSP00000003302': 'USP28', '9606.ENSP00000004531': 'SLC7A2', '9606.ENSP00000004982': 'HSPB6', '9606.ENSP00000005178': 'PDK4', '9606.ENSP00000005226': 'USH1C', '9606.ENSP00000005257': 'RALA', '9606.ENSP00000005260': 'BAIAP2L1', '9606.ENSP00000005284': 'CACNG3', '9606.ENSP00000005286': 'TMEM132A', '9606.ENSP00000005340': 'DVL2', '9606.ENSP00000005386': 'RPAP3', '9606.ENSP00000005587': 'SKAP2', '9606.ENSP00000005995': 'PRSS21', '9606.ENSP00000006015': 'HOXA11', '9606.ENSP00000006053': 'CX3CL1', '9606.ENSP00000006275': 'TRAPPC6A', '9606.ENSP00000006526': 'WDR54', '9606.ENSP00000006658': 'SPATA20', '9606.ENSP00000006724': 'CEACAM7', '9606.ENSP00000006777'

## Part2: Map Ensembl IDs of proteins in the STRING to HGNC symbols

In [48]:
string_df = pd.read_csv("../data/STRING/9606.protein.links.v12.0.min400.onlyAB.tsv", sep="\t")
print(string_df.shape)
string_df.head()

(918440, 3)


,protein1,protein2,combined_score
0,9606.ENSP00000000233,9606.ENSP00000493357,471
1,9606.ENSP00000000233,9606.ENSP00000371175,594
2,9606.ENSP00000000233,9606.ENSP00000354878,513
3,9606.ENSP00000000233,9606.ENSP00000310226,648
4,9606.ENSP00000000233,9606.ENSP00000258098,501


In [49]:
# Map the original string df to HGNC symbols
string_mapped = string_df.copy()
string_mapped["protein1"] = string_df["protein1"].map(protein_id_to_symbol)
string_mapped["protein2"] = string_df["protein2"].map(protein_id_to_symbol)

In [50]:
# Verify the mapping
print(string_mapped.shape)
string_mapped.head()

(918440, 3)


,protein1,protein2,combined_score
0,ARF5,CYTH2,471
1,ARF5,GGA1,594
2,ARF5,KIF21A,513
3,ARF5,RAB1B,648
4,ARF5,RAB11FIP5,501


In [51]:
# Upload our filtered gene pool
gene_pool =pd.read_csv("../data/GSE252168/IG_results_v2", sep="\t", usecols=[0])
# Get the gene symbols as a list
gene_pool = list(gene_pool["Unnamed: 0"])
print(gene_pool)

['A1BG', 'RBFOX1', 'A3GALT2', 'AADAC', 'AADACL4', 'PTGES3L-AARSD1', 'AATK', 'AATK', 'ABCB9', 'ABCC10', 'ABCC4', 'ABCE1', 'ABCE1', 'ABCF2', 'ABCF2', 'ABCF3', 'ABCG1', 'ABCG4', 'ABHD10', 'ABHD14A', 'ABHD14B', 'ABHD2', 'ABHD2', 'ABHD8', 'ABLIM1', 'ABLIM1', 'ABR', 'ABR', 'ABTB1', 'ACACB', 'ACAD10', 'ACAD11', 'ACAD8', 'ACADM', 'ACAP2', 'ACAT1', 'ACAT2', 'ACBD6', 'ASIC3', 'ACCS', 'SDHAF3', 'ACOT2', 'ACOT4', 'ACOT7', 'ACOT9', 'ACOX1', 'ACPP', 'ACRBP', 'ACRC', 'ACSBG1', 'ACSL5', 'ACSS2', 'ACSS2', 'ACTB', 'ACTB', 'ACTB', 'ACTG1', 'ACTL6A', 'ACTN1', 'ACTN4', 'ACTR1B', 'ACTR5', 'ACTR6', 'ACVR1B', 'ACY1', 'ACYP1', 'ACYP2', 'ADA', 'ADAM10', 'ADAM15', 'ADAM19', 'ADAMTS1', 'ADAMTS7', 'ADAR', 'ADAR', 'ADARB1', 'ADARB1', 'ADARB2', 'ADAT1', 'ADAT2', 'ADAT3', 'ADCK1', 'COQ8B', 'ADCY9', 'ADD1', 'ADHFE1', 'ADHFE1', 'ADI1', 'ADIPOR1', 'ADIPOR1', 'ADM', 'ADNP2', 'ADPGK', 'ADSL', 'AEBP1', 'AEN', 'AES', 'AFAP1', 'AFAP1L2', 'AFF1', 'AFF4', 'AFG3L2', 'AFMID', 'AFTPH', 'AGA', 'AGFG2', 'AGGF1', 'AGK', 'AGL', 'AGMA

In [ ]:
# Filter the mapped STRING df for our filtered gene pool
string_mapped_filt = string_mapped[string_mapped["protein1"].isin(gene_pool)]
string_mapped_filt = string_mapped[string_mapped["protein2"].isin(gene_pool)]

In [55]:
# Check the final STRING df size
print(string_mapped_filt.shape)
string_mapped_filt.head()

(270493, 3)


,protein1,protein2,combined_score
3,ARF5,RAB1B,648
6,ARF5,RAB6A,415
9,ARF5,GNB2,563
12,ARF5,MAPK8IP3,451
21,ARF5,KDELR2,581


In [ ]:
# # Save the final filtered version of STRING
# string_mapped_filt.to_csv("../data/STRING/PPI_filtered.tsv", sep="\t", index=False)

## Part 3: Construct filtered PPI via Network X

In [4]:
PPI_df = pd.read_csv("../data/STRING/PPI_filtered.tsv", sep="\t")

In [ ]:
for _, row iteerorws

In [ ]:
# # Check the overall PPI size
# print(f"The number of nodes in the PPI: {nx.number_of_nodes(G)}")
# print(f"The number of nodes in the PPI: {nx.number_of_edges(G)}")

The number of nodes in the PPI: 16733
The number of nodes in the PPI: 270493
